## Transformer Model for translation English -> Finnish.

In [21]:
import keras
import tensorflow as tf
import numpy as np
from keras import layers
from keras import ops
from keras.saving import register_keras_serializable


2025-04-25 15:29:01.532641: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-25 15:29:01.533197: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-25 15:29:01.535370: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-25 15:29:01.540843: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745584141.549635   44789 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745584141.55

Load the finnish to english translation text file and preprocess the text.

In [22]:
text_file = "fin-eng/fin.txt"

with open(text_file, encoding='utf-8') as f:
    lines = f.read().split("\n")[:-1]
text_pairs = []
for line in lines:
    english, finnish, rest = line.split("\t")
    finnish = "[start] " + finnish + " [end]"
    text_pairs.append((english, finnish))

print(text_pairs[:10])

[('Go.', '[start] Mene. [end]'), ('Hi.', '[start] Moro! [end]'), ('Hi.', '[start] Terve. [end]'), ('Run!', '[start] Juokse! [end]'), ('Run!', '[start] Juoskaa! [end]'), ('Run.', '[start] Juokse. [end]'), ('Who?', '[start] Kuka? [end]'), ('Wow!', '[start] Mahtavaa! [end]'), ('Wow!', '[start] Siistiä! [end]'), ('Wow!', '[start] Vau! [end]')]


Separate the data into train, validation & test.

In [ ]:
import random
random.shuffle(text_pairs)
num_val_samples = int(0.15 * len(text_pairs))
num_train_samples = len(text_pairs) - 2 * num_val_samples
train_pairs = text_pairs[:num_train_samples]
val_pairs = text_pairs[num_train_samples:num_train_samples + num_val_samples]
test_pairs = text_pairs[num_train_samples + num_val_samples:]

The `custom_standardization` function prepares text input for the translation model by:

- Converting all text to lowercase
- Removing most punctuation characters
- Preserving square brackets (which are used for special tokens like [start] and [end])

In [ ]:
import string
import re

strip_chars = string.punctuation
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", "")

## Text Vectorization

The code configures two TextVectorization layers for processing the English and Finnish text:

- Sets a vocabulary size of 15,000 tokens for both languages
- Limits sequence length to 20 tokens for inputs
- Uses sequence length of 21 for target outputs (to include end token)
- Converts text to integer token sequences
- Adapts each vectorization layer to its respective language's vocabulary

This preprocessing pipeline transforms raw text into numerical sequences that the transformer model can process, with each word represented by an integer ID from the vocabulary.

In [25]:
vocab_size = 15000
sequence_length = 20

source_vectorization = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)

target_vectorization = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    # standardize=custom_standardization,
)


train_english_texts = [pair[0] for pair in train_pairs]
train_finnish_texts = [pair[1] for pair in train_pairs]
source_vectorization.adapt(train_english_texts)
target_vectorization.adapt(train_finnish_texts)




2025-04-25 15:29:02.578401: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## Dataset Preparation

The code creates TensorFlow datasets for the translation model:

- Sets batch size to 64 samples
- Implements teacher forcing technique for sequence generation:
  - Inputs include source English text and partial Finnish translations
  - Targets are the next Finnish tokens to predict
- Applies performance optimizations:
  - Parallel processing (4 threads)
  - Data shuffling with 2048 buffer size
  - Prefetching (16 batches)
  - Caching for faster training

In [26]:
batch_size = 64

def format_dataset(eng, fin):
    eng = source_vectorization(eng)
    fin = target_vectorization(fin)
    return ({
        "english": eng,
        "finnish": fin[:, :-1],
    }, fin[:, 1:])

def make_dataset(pairs):
    eng_texts, fin_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    fin_texts = list(fin_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, fin_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset,  num_parallel_calls=4)
    return dataset.shuffle(2048).prefetch(16).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

for inputs, targets in train_ds.take(1):
    print(f"inputs['english'].shape: {inputs['english'].shape}")
    print(f"inputs['finnish'].shape: {inputs['finnish'].shape}")

inputs['english'].shape: (64, 20)
inputs['finnish'].shape: (64, 20)


2025-04-25 15:29:02.974985: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2025-04-25 15:29:02.975401: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [27]:
@register_keras_serializable()
class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.attention_2 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.dense_proj = keras.Sequential([
            layers.Dense(dense_dim, activation="relu"),
            layers.Dense(embed_dim),]
            )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.supports_masking = True

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "dense_dim": self.dense_dim,
        })
        return config

    def get_casual_attention_mask(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = tf.range(sequence_length)[:, tf.newaxis]
        j = tf.range(sequence_length)
        mask = tf.cast(i >= j, dtype="int32")
        mask = tf.reshape(mask, (1, input_shape[1], input_shape[1]))  
        mult = tf.concat([tf.expand_dims(batch_size, -1), tf.constant([1, 1], dtype=tf.int32)], axis=0)
        return tf.tile(mask, mult)
    
    def call(self, inputs, encoder_outputs, mask=None):
        # Create a causal mask for self-attention
        causal_mask = self.get_casual_attention_mask(inputs)

        # Combine with padding mask if provided
        if mask is not None:
            padding_mask = tf.cast(mask[:, tf.newaxis, :], dtype="int32")
            padding_mask = tf.minimum(padding_mask, causal_mask)
        else:
            padding_mask = causal_mask

        # Self-attention (decoder attends to previous tokens)
        attention_output_1 = self.attention_1(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=causal_mask
        )
        attention_output_1 = self.layernorm_1(inputs + attention_output_1)

        # Cross-attention (decoder attends to encoder outputs)
        attention_output_2 = self.attention_2(
            query=attention_output_1,
            value=encoder_outputs,
            key=encoder_outputs,
            attention_mask=padding_mask
        )
        attention_output_2 = self.layernorm_2(attention_output_1 + attention_output_2)

        # Feed-forward network (dense projection)
        proj_output = self.dense_proj(attention_output_2)
        
        # Final layer normalization and return
        return self.layernorm_3(attention_output_2 + proj_output)



In [28]:
@register_keras_serializable()
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(dense_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, mask=None):
        if mask is not None:
            padding_mask = ops.cast(mask[:, None, :], dtype="int32")
        else:
            padding_mask = None

        attention_output = self.attention(
            query=inputs, value=inputs, key=inputs, attention_mask=padding_mask
        )
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "dense_dim": self.dense_dim,
                "num_heads": self.num_heads,
            }
        )
        return config

In [29]:
@register_keras_serializable()
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=embed_dim
        )
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

    def call(self, inputs):
        length = ops.shape(inputs)[-1]
        positions = ops.arange(0, length, 1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return ops.not_equal(inputs, 0)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "sequence_length": self.sequence_length,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim,
            }
        )
        return config

## Model Architecture

- Configuration parameters:
  - 256-dimensional embeddings
  - 2048-dimensional feed-forward networks
  - 8 attention heads

- Encoder-Decoder Architecture:
  - **Encoder**: Processes English input text
    - Converts tokens to embeddings with positional information
    - Applies self-attention to capture relationships between words
  
  - **Decoder**: Generates Finnish translation
    - Processes partial Finnish translations
    - Uses causal attention to prevent looking at future tokens
    
- Final output layer transforms decoder features into token probabilities

- The model is trained with:
  - RMSprop optimizer
  - Sparse categorical crossentropy loss
  - 30 training epochs
  - Accuracy as the evaluation metric

In [ ]:
embed_dim = 256
dense_dim = 2048
num_heads = 8

encoder_inputs = keras.Input(shape=(None,), dtype="int64", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(encoder_inputs)
encoder_outputs = TransformerEncoder(embed_dim, dense_dim, num_heads)(x)

decoder_inputs = keras.Input(shape=(None,), dtype="int64", name="finnish")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(decoder_inputs)
x = TransformerDecoder(embed_dim, dense_dim, num_heads)(x, encoder_outputs)
x = layers.Dense(vocab_size, activation="softmax")(x)

transformer = keras.Model([encoder_inputs, decoder_inputs], (x))

transformer.compile(
 optimizer="rmsprop",
 loss="sparse_categorical_crossentropy",
 metrics=["accuracy"])
transformer.fit(train_ds, epochs=30, validation_data=val_ds)

Epoch 1/30
791/791 ━━━━━━━━━━━━━━━━━━━━ 262s 329ms/step - accuracy: 0.1427 - loss: 5.0594 - val_accuracy: 0.1589 - val_loss: 3.6028
Epoch 2/30
791/791 ━━━━━━━━━━━━━━━━━━━━ 258s 326ms/step - accuracy: 0.1613 - loss: 3.7319 - val_accuracy: 0.1773 - val_loss: 3.1302
Epoch 3/30
791/791 ━━━━━━━━━━━━━━━━━━━━ 259s 328ms/step - accuracy: 0.1743 - loss: 3.3097 - val_accuracy: 0.1855 - val_loss: 2.9248
Epoch 4/30
791/791 ━━━━━━━━━━━━━━━━━━━━ 257s 325ms/step - accuracy: 0.1838 - loss: 3.0505 - val_accuracy: 0.1894 - val_loss: 2.8398
Epoch 5/30
791/791 ━━━━━━━━━━━━━━━━━━━━ 257s 325ms/step - accuracy: 0.1900 - loss: 2.8895 - val_accuracy: 0.1925 - val_loss: 2.7950
Epoch 6/30
791/791 ━━━━━━━━━━━━━━━━━━━━ 257s 325ms/step - accuracy: 0.1957 - loss: 2.7627 - val_accuracy: 0.1953 - val_loss: 2.7710
Epoch 7/30
791/791 ━━━━━━━━━━━━━━━━━━━━ 257s 325ms/step - accuracy: 0.2000 - loss: 2.6796 - val_accuracy: 0.1964 - val_loss: 2.7745
Epoch 8/30
791/791 ━━━━━━━━━━━━━━━━━━━━ 257s 325ms/step - accuracy: 0.2033 -

Here the model is saved into a file (Commented out just for testing different models, but the best one should be stored)

In [ ]:
# # Load model with explicit custom_objects parameter
# loaded_model = keras.saving.load_model(
#     "english_finnish_transformer.keras", 
#     custom_objects={
#         "PositionalEmbedding": PositionalEmbedding,
#         "TransformerEncoder": TransformerEncoder,
#         "TransformerDecoder": TransformerDecoder
#     }
# )

Then, test the model with the test dataset. Unfortunately the model doesn't perform great with only about 24% accuracy, but it does get some sentences correctly, but most of them are completely off.

In [ ]:
import numpy as np
fin_vocab = target_vectorization.get_vocabulary()
fin_index_lookup = dict(zip(range(len(fin_vocab)), fin_vocab))
max_decoded_sentence_length = 20

def decode_sequence(input_sequence):
    tokenized_input_sentence = source_vectorization([input_sequence])
    decoded_sentence = "[start]"
    for i in range(max_decoded_sentence_length):
        tokenized_target_sentence = target_vectorization([decoded_sentence])[:, :-1]
        predictions = transformer([tokenized_input_sentence, tokenized_target_sentence])
        sampled_token_index = np.argmax(predictions[0, i, :])
        sampled_token = fin_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence
    
test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(20):
    input_sentence = random.choice(test_eng_texts)
    print(input_sentence)
    print(decode_sequence(input_sentence))

Can you give this to Tom?
[start] voitko kertoa tomille tässä end  end end end end end end end end end end end  voi voi
Can you speak French, too?
[start] osaatko puhua ranskaa end  end end end end end end end end end      
These flowers are beautiful, aren't they?
[start] nämä ovat ne [UNK] eikö vain end  end end end end end end end end end end end ne
Everybody makes mistakes.
[start] kaikki ovat loppuun end  end end end end end end kaikki end kaikki end kaikki kaikki loppuun kaikki loppuun
He does not study hard enough.
[start] hän ei tekee [UNK] paljon tarpeeksi kovasti end  end end end end end  tekee  end  
Tom is the black sheep of his family.
[start] tomi on minun [UNK] [UNK] [UNK] end  end end end end end end end     
I broke Tom's nose.
[start] [UNK] tomin kädet end  end end end end end end end end end end end end end  end
I don't recommend that.
[start] en ota sitä end  end end end end end end end end end end end  sitä end 
Don't tell me you were worried.
[start] Älä kerro min